In [0]:
# ===========================================
# Notebook Name:
# 02_validate_daily_pipeline
#
# Purpose:
# Gate for the Daily Pokemon Lakehouse
# Pipeline (run_if: ALL_DONE, placed after
# the Gold rebuild tasks). Reports how many
# new tournaments / results / decks were
# produced by this run and fails the Job if
# too many scrape errors occurred.
#
# Input:
# pokemon.bronze.scrape_log
# pokemon.silver.tournaments
# pokemon.silver.tournament_results
# pokemon.silver.decks
#
# Output:
# None (validation only; raises on failure)
# ===========================================

In [0]:
from pyspark.sql import functions as F

SCRAPE_LOG_TABLE = "pokemon.bronze.scrape_log"
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"
TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)
DECKS_TABLE = "pokemon.silver.decks"

ERROR_STATUS_VALUES = [
    "error",
    "failed",
    "failure",
    "http_error",
]

MAX_ALLOWED_ERROR_COUNT = 0

started_at_iso = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="started_at_iso",
    default=None,
    debugValue=None,
)

if started_at_iso is None:
    raise ValueError(
        "started_at_iso not found in "
        "taskValues. Run "
        "00_initialize_pipeline_run first."
    )

started_at = F.to_timestamp(
    F.lit(started_at_iso)
)

print("Run started at:", started_at_iso)

In [0]:
new_tournament_count = (
    spark.table(TOURNAMENTS_TABLE)
    .filter(
        F.col("first_seen_at") >= started_at
    )
    .count()
)

new_result_count = (
    spark.table(TOURNAMENT_RESULTS_TABLE)
    .filter(
        F.col("updated_at") >= started_at
    )
    .count()
)

new_deck_count = (
    spark.table(DECKS_TABLE)
    .filter(
        F.col("updated_at") >= started_at
    )
    .count()
)

print(
    "New tournaments      :",
    new_tournament_count,
)
print(
    "New/updated results  :",
    new_result_count,
)
print(
    "New/updated decks    :",
    new_deck_count,
)

In [0]:
error_df = (
    spark.table(SCRAPE_LOG_TABLE)
    .filter(
        F.col("scraped_at") >= started_at
    )
    .filter(
        F.lower(F.col("status")).isin(
            *ERROR_STATUS_VALUES
        )
    )
)

error_count = error_df.count()

print("Scrape errors:", error_count)

if error_count > 0:
    display(
        error_df
        .groupBy("source_type", "status")
        .count()
        .orderBy("source_type", "status")
    )

    display(
        error_df.select(
            "source_type",
            "source_id",
            "status",
            "error_message",
            "scraped_at",
        )
        .orderBy(F.col("scraped_at").desc())
        .limit(50)
    )

In [0]:
if error_count > MAX_ALLOWED_ERROR_COUNT:
    raise ValueError(
        f"{error_count} scrape error(s) "
        "detected during this run, above the "
        f"allowed threshold of "
        f"{MAX_ALLOWED_ERROR_COUNT}. "
        "See the error breakdown above."
    )

print(
    "Validation passed: daily pipeline "
    "completed within the error threshold "
    f"(new_tournaments={new_tournament_count}, "
    f"new_results={new_result_count}, "
    f"new_decks={new_deck_count}, "
    f"errors={error_count})"
)